# Logistic Regression from Scratch with PyTorch

In [ ]:
import numpy as np
import torch

from sklearn import compose, datasets, linear_model, metrics, model_selection
from sklearn import pipeline, preprocessing

## Load the CovType dataset

In [ ]:
covtype_dataset = datasets.fetch_covtype(
    as_frame=True
)

In [ ]:
print(covtype_dataset["DESCR"])

In [ ]:
covtype_features_df = covtype_dataset["data"]
covtype_target_df = (
    covtype_dataset.get("target")
                   .to_frame()
)

In [ ]:
covtype_features_df.info()

In [ ]:
_ = (
    covtype_target_df.loc[:, "Cover_Type"]
                     .value_counts()
                     .sort_index()
                     .plot(kind="bar")
)

## Train/Val Split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, val_features_df, train_target_df, val_target_df = (
    model_selection.train_test_split(
        covtype_features_df,
        covtype_target_df,
        test_size=0.20,
        shuffle=True,
        stratify=covtype_target_df,
        random_state=RANDOM_STATE
    )
)


In [ ]:
train_features_df.info()

In [ ]:
val_features_df.info()

## Prepare the data

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


prepare_covtype_features = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            "passthrough",
            compose.make_column_selector(
                pattern="^Wilderness_Area_|^Soil_Type_"
            )
        ),
        force_int_remainder_cols=False,
        n_jobs=-1,
        remainder=preprocessing.QuantileTransformer(
            output_distribution="normal",
            random_state=RANDOM_STATE,
        )
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
    )
)

prepare_covtype_target = pipeline.make_pipeline(
    preprocessing.OrdinalEncoder(
        categories=[
            [1, 2, 3, 4, 5, 6, 7]
        ],
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    ),
    preprocessing.FunctionTransformer(
        func=torch.squeeze,
    )
)



In [ ]:
X_train = prepare_covtype_features.fit_transform(train_features_df)
X_val = prepare_covtype_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_covtype_target.fit_transform(train_target_df)
y_val = prepare_covtype_target.transform(val_target_df)


In [ ]:
print(y_train.shape)
print(y_train.dtype)

print(y_val.shape)
print(y_val.dtype)

## Logistic Regression using Tensors

### Initialize parameters

In [ ]:
prng = torch.manual_seed(42)

n_features = X_train.size(1)
n_classes = y_train.unique().size(0)

weights = torch.randn((n_features, n_classes), requires_grad=True)
bias = torch.zeros(1, n_classes, requires_grad=True)

In [ ]:
print(weights.shape)
print(bias.shape)

### Define our model and loss functions

In [ ]:
def model_fn(X):
    return X @ weights + bias


def softmax_activation_fn(y_pred):
    return torch.exp(y_pred) / torch.sum(torch.exp(y_pred), dim=1, keepdim=True)


def log_softmax_activation_fn(y_pred):
    return torch.log(softmax_activation_fn(y_pred))


def neg_log_likelihood_loss_fn(log_probas_pred, y_true):
    batch_indicies = torch.arange(y_true.size(0))
    target_indices = y_true
    return -torch.mean(log_probas_pred[batch_indicies, target_indices])


def loss_fn(y_pred, y_true):
    log_probas_pred = log_softmax_activation_fn(y_pred)
    return neg_log_likelihood_loss_fn(log_probas_pred, y_true)



### Training using full batch gradient descent

In [ ]:
learning_rate = 0.5
n_epochs = 100

for epoch in range(n_epochs):
    # forward pass
    y_pred = model_fn(X_train)
    train_loss = loss_fn(y_pred, y_train)

    # backward pass
    train_loss.backward()

    # gradient descent step
    with torch.no_grad():
        bias -= learning_rate * bias.grad
        weights -= learning_rate * weights.grad
        bias.grad.zero_()
        weights.grad.zero_()

    # evaluate using the validation data
    with torch.no_grad():
        y_pred = model_fn(X_val)
        val_loss = loss_fn(y_pred, y_val)

    print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")



## Logistic Regression using the Neural Network API

In [ ]:
from torch import nn, optim

### Define our model and loss functions, and our optimizer

In [ ]:
_ = torch.manual_seed(42)

# logistic regression only has a single layer
covtype_model = nn.Linear(
    in_features=n_features,
    out_features=n_classes,
    bias=True,
)

# using the cross entropy loss
cross_entropy_loss = nn.CrossEntropyLoss()

# define our optimizer
learning_rate = 0.5
sgd = optim.SGD(
    covtype_model.parameters(),
    lr=learning_rate
)

### Training using full batch gradient descent

In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs
    ):

    for epoch in range(n_epochs):
        # forward pass
        y_pred = model_fn(X_train)
        train_loss = criterion(y_pred, y_train)

        # backward pass
        train_loss.backward()

        # gradient descent step
        optimizer.step()
        optimizer.zero_grad()

        # evaluate using the validation data
        with torch.no_grad():
            y_pred = model_fn(X_val)
            val_loss = criterion(y_pred, y_val)

        print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")


In [ ]:
train(
    covtype_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100
)

### Compare results with Scikit-Learn SGDClassifier

In [ ]:
sgd_classifier = pipeline.make_pipeline(
    prepare_covtype_features[:-1],
    linear_model.SGDClassifier(
        learning_rate="constant",
        eta0=learning_rate,
        loss="log_loss",
        max_iter=100,
        n_jobs=-1,
        penalty=None,
        random_state=RANDOM_STATE,
    )
)

train_target = train_target_df.loc[:, "Cover_Type"]
_ = sgd_classifier.fit(train_features_df, train_target)

In [ ]:
train_prediction = sgd_classifier.predict_proba(train_features_df)
train_log_loss = metrics.log_loss(
    train_target,
    train_prediction
)


val_target = val_target_df.loc[:, "Cover_Type"]
val_prediction = sgd_classifier.predict_proba(val_features_df)
val_log_loss = metrics.log_loss(
    val_target,
    val_prediction
)

print(f"Training loss: {train_log_loss: .4f}, Validation loss: {val_log_loss: .4f}")

## Exercise

Load the breast cancer dataset using the code in the cell below. Prepare the data and then train a logistic regression model using PyTorch Neural Network API.

In [ ]:
breast_cancer_dataset = datasets.load_breast_cancer(
    as_frame=True,
)

In [ ]:
print(breast_cancer_dataset["DESCR"])

In [ ]:
breast_cancer_features_df = breast_cancer_dataset["data"]
breast_cancer_target = breast_cancer_dataset["target"]

In [ ]:
breast_cancer_features_df.info()

In [ ]:
breast_cancer_features_df.describe()

In [ ]:
_ = breast_cancer_target.hist()

In [ ]:
# INSERT YOUR CODE HERE!

### Solution

In [ ]:
train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        breast_cancer_features_df,
        breast_cancer_target,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=breast_cancer_target,
        test_size=0.20,
    )
)


In [ ]:
def dataframe_to_tensor(df, dtype=torch.float32):
    arr = df.to_numpy()
    return array_to_tensor(arr, dtype)


def series_to_tensor(s, dtype=torch.float32):
    df = s.to_frame()
    return dataframe_to_tensor(df, dtype)


n_samples, _ = train_features_df.shape
prepare_breast_cancer_features = pipeline.make_pipeline(
    preprocessing.QuantileTransformer(
      n_quantiles=n_samples,
      output_distribution="normal",
      random_state=RANDOM_STATE
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_breast_cancer_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    )
)

In [ ]:
X_train = prepare_breast_cancer_features.fit_transform(train_features_df)
X_val = prepare_breast_cancer_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_breast_cancer_target.fit_transform(train_target)
y_val = prepare_breast_cancer_target.transform(val_target)


In [ ]:
print(y_train.shape)
print(y_val.shape)

In [ ]:
_ = torch.manual_seed(42)

# logistic regression only has a single layer
n_features = X_train.size(1)
n_classes = y_train.unique().size(0)

breast_cancer_model = nn.Linear(
    in_features=n_features,
    out_features=n_classes,
    bias=True,
)

# continue using the cross-entropy loss
cross_entropy_loss = nn.CrossEntropyLoss()

# define our optimizer
learning_rate = 1e-2
sgd = optim.SGD(
    breast_cancer_model.parameters(),
    lr=learning_rate
)

# train the model
train(
    breast_cancer_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100
)

## Exercise (Optional)

Confirm that your results above are similar to those obtained using SGDRegressor from Scikit-Learn.


In [ ]:
# INSERT YOUR CODE HERE!

### Solution

In [ ]:
sgd_classifier = pipeline.make_pipeline(
    prepare_breast_cancer_features[:-1],
    linear_model.SGDClassifier(
        learning_rate="constant",
        eta0=learning_rate,
        loss="log_loss",
        max_iter=100,
        n_jobs=-1,
        penalty=None,
        random_state=RANDOM_STATE,
    )
)

_ = sgd_classifier.fit(train_features_df, train_target)

In [ ]:
train_prediction = sgd_classifier.predict_proba(train_features_df)
train_log_loss = metrics.log_loss(
    train_target,
    train_prediction
)


val_prediction = sgd_classifier.predict_proba(val_features_df)
val_log_loss = metrics.log_loss(
    val_target,
    val_prediction
)

print(f"Training loss: {train_log_loss: .4f}, Validation loss: {val_log_loss: .4f}")